In [1]:
from faker import Faker
import random
import psycopg2
from psycopg2 import connect
from dotenv import load_dotenv
import os

# Khai báo biến môi trường
load_dotenv()
DB_NAME = os.getenv("DB_NAME")
USER = os.getenv("USER")
PASS = os.getenv("PASSWORD")

In [2]:
def feed_users_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Users (PostgreSQL).
    """
    fake = Faker('en_US')
    fake_data = []

    for _ in range(n):
        full_name = fake.name()
        
        # Đảm bảo email là duy nhất (UNIQUE)
        email = fake.unique.email()
        
        # Mật khẩu thực tế lưu trong DB phải là chuỗi đã băm (hashed)
        # Sử dụng sha256 của Faker để giả lập một chuỗi hash 64 ký tự hợp lệ
        password_hash = fake.sha256()
        
        # Số điện thoại: Dùng numerify để kiểm soát độ dài dưới 20 ký tự, tránh lỗi
        phone = fake.numerify(text='+1-###-###-####')
        
        # Ngày tạo tài khoản: Random trong vòng 2 năm trở lại đây
        created_at = fake.date_time_between(start_date='-2y', end_date='now')

        # Thêm tuple vào danh sách (Lưu ý: Không truyền user_id vì SERIAL tự tăng)
        fake_data.append((full_name, email, password_hash, phone, created_at))

    # Chèn dữ liệu vào bảng (Sử dụng %s cho PostgreSQL)
    cursor.executemany('''
        INSERT INTO Users (full_name, email, password_hash, phone, created_at)
        VALUES (%s, %s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Users.")

In [3]:
def feed_categories_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Categories (PostgreSQL).
    """
    fake = Faker('en_US')
    fake_data = []

    # Cung cấp sẵn một danh sách các danh mục E-commerce thực tế nhất
    real_categories = [
        "Electronics", "Clothing & Accessories", "Home & Kitchen", 
        "Sports & Outdoors", "Books & Audible", "Beauty & Personal Care", 
        "Toys & Games", "Automotive Parts", "Health & Household", 
        "Grocery & Gourmet Food", "Pet Supplies", "Office Products", 
        "Jewelry & Watches", "Furniture", "Tools & Home Improvement"
    ]

    for i in range(n):
        # Ưu tiên lấy tên từ danh sách thực tế trước
        if i < len(real_categories):
            category_name = real_categories[i]
        else:
            # Nếu bạn muốn tạo nhiều hơn số danh mục có sẵn, sinh tên ngẫu nhiên
            # VD: "Tech & Gadgets", "Music & Movies"
            category_name = f"{fake.unique.word().capitalize()} & {fake.word().capitalize()}"
            
        # Tạo một đoạn mô tả ngắn gọn (tối đa 100 ký tự)
        description = fake.text(max_nb_chars=100) 
        
        # Thêm tuple vào danh sách (Không truyền category_id vì là SERIAL)
        fake_data.append((category_name, description))

    # Chèn dữ liệu (Sử dụng placeholder %s của PostgreSQL)
    cursor.executemany('''
        INSERT INTO Categories (category_name, description)
        VALUES (%s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Categories.")

In [4]:
def feed_products_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Products (PostgreSQL).
    """
    fake = Faker('en_US')
    
    # 1. Lấy danh sách category_id hiện có từ bảng Categories
    cursor.execute("SELECT category_id FROM Categories")
    category_records = cursor.fetchall()
    valid_category_ids = [record[0] for record in category_records] if category_records else []
    
    if not valid_category_ids:
        print("Cảnh báo: Bảng Categories trống! Các sản phẩm sẽ có category_id = NULL.")

    fake_data = []

    # Danh sách từ vựng để ghép tên sản phẩm sao cho "chuẩn" E-commerce
    adjectives = ["Smart", "Ergonomic", "Wireless", "Portable", "Premium", "Compact", "Heavy-Duty", "Eco-friendly", "Luxury", "Classic"]
    nouns = ["Headphones", "Monitor", "Keyboard", "Chair", "Backpack", "Bottle", "Speaker", "Desk", "Watch", "Camera"]

    for _ in range(n):
        # Tên sản phẩm: Tính từ + Danh từ + Mã model ngẫu nhiên (Ví dụ: Smart Headphones 1023)
        product_name = f"{random.choice(adjectives)} {random.choice(nouns)} {fake.bothify(text='??-###').upper()}"
        
        # Giá tiền: Từ $5.00 đến $1,500.00
        price = round(random.uniform(5.0, 1500.0), 2)
        
        # Tồn kho: Từ 0 (hết hàng) đến 500
        stock_quantity = random.randint(0, 500)
        
        # Mô tả sản phẩm (Khoảng 2-3 câu)
        description = fake.paragraph(nb_sentences=2)
        
        # Chọn ngẫu nhiên một danh mục hợp lệ
        category_id = random.choice(valid_category_ids) if valid_category_ids else None

        # Không chèn product_id vì là SERIAL (Tự động tăng)
        fake_data.append((product_name, price, stock_quantity, description, category_id))

    # 2. Chèn dữ liệu vào bảng (Dùng %s cho PostgreSQL)
    cursor.executemany('''
        INSERT INTO Products (product_name, price, stock_quantity, description, category_id)
        VALUES (%s, %s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Products.")

In [5]:
def feed_shopping_cart_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Shopping_Cart (PostgreSQL).
    """
    fake = Faker('en_US')
    
    # 1. Lấy danh sách user_id hiện có từ bảng Users
    cursor.execute("SELECT user_id FROM Users")
    user_records = cursor.fetchall()
    valid_user_ids = [record[0] for record in user_records] if user_records else []
    
    if not valid_user_ids:
        print("Cảnh báo: Bảng Users trống! Giỏ hàng sẽ không được gắn với người dùng nào (NULL).")

    fake_data = []

    for _ in range(n):
        # Chọn ngẫu nhiên một người dùng sở hữu giỏ hàng này
        user_id = random.choice(valid_user_ids) if valid_user_ids else None
        
        # Giỏ hàng thường là những thao tác mới đây, random trong vòng 1 tháng qua
        created_at = fake.date_time_between(start_date='-1m', end_date='now')

        # Không chèn cart_id vì là kiểu SERIAL
        fake_data.append((user_id, created_at))

    # 2. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Shopping_Cart (user_id, created_at)
        VALUES (%s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Shopping_Cart.")

In [6]:
def feed_cart_items_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Cart_Items (PostgreSQL).
    """
    # 1. Lấy danh sách cart_id hiện có từ bảng Shopping_Cart
    cursor.execute("SELECT cart_id FROM Shopping_Cart")
    cart_records = cursor.fetchall()
    valid_cart_ids = [record[0] for record in cart_records] if cart_records else []
    
    # 2. Lấy danh sách product_id hiện có từ bảng Products
    cursor.execute("SELECT product_id FROM Products")
    product_records = cursor.fetchall()
    valid_product_ids = [record[0] for record in product_records] if product_records else []
    
    if not valid_cart_ids or not valid_product_ids:
        print("Cảnh báo: Bảng Shopping_Cart hoặc Products đang trống! Không thể chèn dữ liệu vào Cart_Items một cách chính xác.")
        return

    fake_data = []

    for _ in range(n):
        cart_id = random.choice(valid_cart_ids)
        product_id = random.choice(valid_product_ids)
        
        # Số lượng thực tế người dùng hay mua: thường từ 1 đến 3, thi thoảng lên tới 5 hoặc 10
        quantity = random.choices([1, 2, 3, 4, 5, 10], weights=[60, 20, 10, 5, 3, 2], k=1)[0]

        # Không chèn cart_item_id vì là kiểu SERIAL
        fake_data.append((cart_id, product_id, quantity))

    # 3. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Cart_Items (cart_id, product_id, quantity)
        VALUES (%s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Cart_Items.")

In [7]:
def feed_orders_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Orders (PostgreSQL).
    """
    fake = Faker('en_US')
    
    # 1. Lấy danh sách user_id hiện có từ bảng Users
    cursor.execute("SELECT user_id FROM Users")
    user_records = cursor.fetchall()
    valid_user_ids = [record[0] for record in user_records] if user_records else []
    
    if not valid_user_ids:
        print("Cảnh báo: Bảng Users trống! Đơn hàng sẽ có user_id = NULL.")

    fake_data = []
    
    # Các trạng thái đơn hàng thực tế kèm theo trọng số xuất hiện (weights)
    # Đa số đơn hàng sẽ ở trạng thái 'Delivered' (Đã giao) hoặc 'Shipped'
    statuses = ['Pending', 'Processing', 'Shipped', 'Delivered', 'Cancelled', 'Refunded']
    status_weights = [10, 10, 20, 50, 5, 5]

    for _ in range(n):
        user_id = random.choice(valid_user_ids) if valid_user_ids else None
        
        # Ngày đặt hàng: Random trong vòng 1 năm trở lại đây
        order_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        # Tổng tiền: Từ $10.00 đến $2000.00
        total_amount = round(random.uniform(10.0, 2000.0), 2)
        
        # Lấy trạng thái ngẫu nhiên theo trọng số
        status = random.choices(statuses, weights=status_weights, k=1)[0]

        # Không chèn order_id vì là SERIAL
        fake_data.append((user_id, order_date, total_amount, status))

    # 2. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Orders (user_id, order_date, total_amount, status)
        VALUES (%s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Orders.")

In [8]:
def feed_order_items_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Order_Items (PostgreSQL).
    """
    # 1. Lấy danh sách order_id hiện có từ bảng Orders
    cursor.execute("SELECT order_id FROM Orders")
    order_records = cursor.fetchall()
    valid_order_ids = [record[0] for record in order_records] if order_records else []
    
    # 2. Lấy danh sách product_id VÀ price từ bảng Products 
    # Để lưu vào price_at_purchase cho chuẩn logic thực tế
    cursor.execute("SELECT product_id, price FROM Products")
    product_records = cursor.fetchall()
    
    if not valid_order_ids or not product_records:
        print("Cảnh báo: Bảng Orders hoặc Products đang trống! Không thể chèn dữ liệu vào Order_Items.")
        return

    fake_data = []

    for _ in range(n):
        order_id = random.choice(valid_order_ids)
        
        # Chọn ngẫu nhiên một sản phẩm và lấy luôn giá của nó
        selected_product = random.choice(product_records)
        product_id = selected_product[0]
        base_price = selected_product[1]
        
        # Số lượng mua (thường mua 1-3 cái)
        quantity = random.choices([1, 2, 3, 4, 5], weights=[60, 20, 10, 5, 5], k=1)[0]
        
        # Giá tại thời điểm mua (Có thể thi thoảng được giảm giá nhẹ so với giá gốc)
        # Random giảm giá từ 0% đến 10% cho thực tế
        discount = random.uniform(0.9, 1.0)
        price_at_purchase = round(float(base_price) * discount, 2)

        # Không chèn order_item_id vì là SERIAL
        fake_data.append((order_id, product_id, quantity, price_at_purchase))

    # 3. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Order_Items (order_id, product_id, quantity, price_at_purchase)
        VALUES (%s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Order_Items.")

In [9]:
def feed_payments_data(n, cursor):
    """
    Hàm sinh dữ liệu CƠ BẢN và THỰC TẾ cho bảng Payments (PostgreSQL).
    """
    fake = Faker('en_US')
    
    # 1. Lấy order_id và total_amount từ bảng Orders
    # Việc lấy total_amount giúp số tiền thanh toán khớp với giá trị đơn hàng
    cursor.execute("SELECT order_id, total_amount FROM Orders")
    order_records = cursor.fetchall()
    
    if not order_records:
        print("Cảnh báo: Bảng Orders trống! Không thể chèn dữ liệu vào Payments.")
        return

    fake_data = []
    
    # Các phương thức thanh toán và trọng số (Thẻ tín dụng và ví điện tử phổ biến nhất)
    methods = ['Credit Card', 'PayPal', 'Cash on Delivery', 'Bank Transfer']
    method_weights = [55, 30, 10, 5]
    
    # Trạng thái thanh toán
    statuses = ['Completed', 'Pending', 'Failed', 'Refunded']
    status_weights = [80, 10, 5, 5]

    for _ in range(n):
        # Chọn ngẫu nhiên một đơn hàng
        selected_order = random.choice(order_records)
        order_id = selected_order[0]
        amount = selected_order[1]  # Gán chính xác số tiền cần thanh toán của đơn hàng đó
        
        payment_method = random.choices(methods, weights=method_weights, k=1)[0]
        status = random.choices(statuses, weights=status_weights, k=1)[0]
        
        # Ngày giao dịch (Transaction date)
        transaction_date = fake.date_time_between(start_date='-1y', end_date='now')

        # Không chèn payment_id vì là SERIAL
        fake_data.append((order_id, payment_method, amount, status, transaction_date))

    # 2. Chèn dữ liệu vào bảng
    cursor.executemany('''
        INSERT INTO Payments (order_id, payment_method, amount, status, transaction_date)
        VALUES (%s, %s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Payments.")

In [10]:
def feed_shipping_data(n, cursor):
    """
    Hàm sinh dữ liệu cho bảng Shipping (PostgreSQL).
    """
    fake = Faker('en_US')
    
    cursor.execute("SELECT order_id FROM Orders")
    order_records = cursor.fetchall()
    valid_order_ids = [record[0] for record in order_records] if order_records else []

    fake_data = []
    
    # Trạng thái giao hàng
    statuses = ['Preparing', 'Shipped', 'In Transit', 'Delivered']
    weights = [15, 20, 25, 40]

    for _ in range(n):
        order_id = random.choice(valid_order_ids) if valid_order_ids else None
        
        # Địa chỉ giao hàng: Thay thế dấu xuống dòng bằng dấu phẩy cho gọn
        shipping_address = fake.address().replace('\n', ', ')
        
        # Tạo mã vận đơn thực tế (Ví dụ: FEDEX-123456789)
        carrier = random.choice(['FEDEX', 'UPS', 'DHL', 'USPS'])
        tracking_number = f"{carrier}-{fake.unique.numerify(text='#########')}"
        
        status = random.choices(statuses, weights=weights, k=1)[0]
        
        # LOGIC THỰC TẾ: Nếu đơn đang 'Preparing' (Chuẩn bị) thì chưa thể có shipped_date (NULL)
        if status == 'Preparing':
            shipped_date = None
        else:
            shipped_date = fake.date_time_between(start_date='-1m', end_date='now')

        fake_data.append((order_id, shipping_address, tracking_number, status, shipped_date))

    cursor.executemany('''
        INSERT INTO Shipping (order_id, shipping_address, tracking_number, status, shipped_date)
        VALUES (%s, %s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Shipping.")

In [11]:
def feed_reviews_data(n, cursor):
    """
    Hàm sinh dữ liệu cho bảng Reviews (PostgreSQL).
    """
    fake = Faker('en_US')
    
    cursor.execute("SELECT user_id FROM Users")
    valid_user_ids = [r[0] for r in cursor.fetchall()]
    
    cursor.execute("SELECT product_id FROM Products")
    valid_product_ids = [r[0] for r in cursor.fetchall()]

    fake_data = []

    for _ in range(n):
        user_id = random.choice(valid_user_ids) if valid_user_ids else None
        product_id = random.choice(valid_product_ids) if valid_product_ids else None
        
        # LOGIC THỰC TẾ: Đa số người mua hàng sẽ đánh giá 4 hoặc 5 sao
        rating = random.choices([1, 2, 3, 4, 5], weights=[5, 5, 10, 30, 50], k=1)[0]
        
        # Thường thì 20% người dùng chấm sao nhưng lười không để lại bình luận (NULL)
        comment = fake.sentence(nb_words=12) if random.random() > 0.2 else None
        
        review_date = fake.date_time_between(start_date='-1y', end_date='now')

        fake_data.append((user_id, product_id, rating, comment, review_date))

    cursor.executemany('''
        INSERT INTO Reviews (user_id, product_id, rating, comment, review_date)
        VALUES (%s, %s, %s, %s, %s)
    ''', fake_data)
    
    print(f"Đã thêm thành công {n} dòng vào bảng Reviews.")

In [15]:
conn = connect(
    host="localhost",
    port="5432",
    database=DB_NAME,
    user=USER,
    password=PASS
)

cursor = conn.cursor()
cursor.execute("SELECT * FROM users")
print(cursor.fetchall())
conn.close()

[(1, 'Billy Miller', 'fmorgan@example.net', '487020bbb22009dffc7631f0f8395330de859710c1146aa15f64ca0fb9769bdc', '+1-107-814-3152', datetime.datetime(2025, 2, 5, 8, 21)), (2, 'Miss Catherine Knox', 'johnny80@example.org', '02591188eee46e7b153d15fd0f1442469c02ee6af2d8550baac9a5fea1d70467', '+1-685-628-8610', datetime.datetime(2024, 6, 23, 16, 53, 29)), (3, 'Patricia Taylor MD', 'tylerdavid@example.org', '01d45686d30f288ea379d603635e9348d74eeca2235da633c276125f2d9b7bab', '+1-644-580-9236', datetime.datetime(2025, 12, 18, 9, 1, 17)), (4, 'John Fletcher Jr.', 'jeromerivers@example.com', '0bbc9bb2ac82cf9693ae77541396f25424237148f35bf456aeecbca4ed9b8be8', '+1-755-267-3980', datetime.datetime(2024, 10, 31, 18, 57, 23)), (5, 'Troy Sanchez', 'gtaylor@example.org', '18a2c5361338d4ada7f8e106e49e71bc68b1e720a3fec0d2a5dbe531af1a52cb', '+1-000-419-6583', datetime.datetime(2024, 9, 2, 12, 32, 27)), (6, 'Alexis Moon', 'roblesnicole@example.org', '496623661eeb71ac39503bfa01d8b1787dc8be5d973d0b6895d4c460

In [ ]:
# 1. CẤU HÌNH KẾT NỐI POSTGRESQL
# Hãy thay đổi các thông tin này cho khớp với database của bạn
try:
    # Mở kết nối
    print("- Đang mở kết nối đến database nghiệp vụ...")
    conn = connect(
        host="localhost",
        port="5432",
        database=DB_NAME,
        user='postgres',
        password='YOUR ADMIN PASSWORD'
    )
    cursor = conn.cursor()
    print("=> Mở kết nối đến database nghiệp vụ thành công !")
    print("Bắt đầu tạo dữ liệu...\n")

    # 2. GỌI CÁC HÀM THEO ĐÚNG THỨ TỰ RÀNG BUỘC KHÓA NGOẠI (FOREIGN KEYS)
    # Nhóm 1: Các bảng không phụ thuộc
    feed_users_data(50, cursor)
    feed_categories_data(15, cursor)
    
    # Nhóm 2: Các bảng phụ thuộc cấp 1
    feed_products_data(100, cursor)
    feed_shopping_cart_data(30, cursor)
    feed_orders_data(150, cursor)
    
    # Nhóm 3: Các bảng phụ thuộc cấp 2 (chi tiết và liên kết nhiều bảng)
    feed_cart_items_data(80, cursor)
    feed_order_items_data(350, cursor)
    feed_payments_data(150, cursor)
    feed_shipping_data(150, cursor)
    feed_reviews_data(120, cursor)

    # 3. LƯU TOÀN BỘ DỮ LIỆU
    # Chỉ commit 1 lần duy nhất ở cuối để tối ưu tốc độ I/O ổ cứng
    conn.commit()
    print("\nHOÀN TẤT! Đã lưu toàn bộ dữ liệu giả vào cơ sở dữ liệu.")

except (Exception, psycopg2.DatabaseError) as error:
    # Nếu có bất kỳ lỗi gì (sai kiểu dữ liệu, thiếu bảng, v.v.), hủy toàn bộ thao tác
    if conn:
        conn.rollback()
    print(f"\n[LỖI] Đã xảy ra lỗi trong quá trình tạo dữ liệu: {error}")
    print("Đã hoàn tác (rollback) toàn bộ thay đổi để bảo vệ Database.")

finally:
    # 4. ĐÓNG KẾT NỐI AN TOÀN
    if conn:
        cursor.close()
        conn.close()
        print("Đã đóng kết nối PostgreSQL.")

- Đang mở kết nối đến database nghiệp vụ...
=> Mở kết nối đến database nghiệp vụ thành công !
Bắt đầu tạo dữ liệu...

Đã thêm thành công 50 dòng vào bảng Users.
Đã thêm thành công 15 dòng vào bảng Categories.
Đã thêm thành công 100 dòng vào bảng Products.
Đã thêm thành công 30 dòng vào bảng Shopping_Cart.
Đã thêm thành công 150 dòng vào bảng Orders.
Đã thêm thành công 80 dòng vào bảng Cart_Items.
Đã thêm thành công 350 dòng vào bảng Order_Items.
Đã thêm thành công 150 dòng vào bảng Payments.
Đã thêm thành công 150 dòng vào bảng Shipping.
Đã thêm thành công 120 dòng vào bảng Reviews.

HOÀN TẤT! Đã lưu toàn bộ dữ liệu giả vào cơ sở dữ liệu.
Đã đóng kết nối PostgreSQL.
